In [ ]:
import pandas as pd
from gnome_dreams_oer_screening import RelationalFrame, OER_catalyst_study

### Creat RelationalFrame -> Add PD Table -> Build Relations

**Table Selection**\
RelationalFrame.<table\>\
\
**View Table**\
RelationalFrame.<table\>.df\
\
**View Relations**\
RelationalFrame.<table\>.relations\
\
**Select Row**\
RelationalFrame.<table\>[primary_key\]\
\
**Select Value of Row**\
RelationalFrame.<table\>[primary_key\].<column\>\
\
**Select SubTable(s)**\
RelationalFrame.<table\>[primary_key\].<subTable(relation)\>\
\
**Add Row**\
RelationalFrame.<table\>.add_row(Dict/pd)\
RelationalFrame.<table\>[primary_key\].<subTable(relation)\>.add_row(Dict/pd)

In [7]:
# creat an arbitrary class

class study_object:
    def __init__(self, id, name):
        self.id = id
        self.name = name

tmpStudyObj1 = study_object(1, "test_study1")
tmpStudyObj2 = study_object(2, "test_study2")

In [8]:
ExpLog = RelationalFrame(root="candidates")

In [9]:
study_df = pd.DataFrame(columns=["candidate_id", "reason_or_hypothesis", "notes", "study_obj", "OHDone", "idealOverPotential"])
ExpLog.add_table("candidates", study_df, pk="candidate_id")

<Table candidates rows=0 pk=candidate_id>

In [10]:
ExpLog.candidates.df

,candidate_id,reason_or_hypothesis,notes,study_obj,OHDone,idealOverPotential


In [11]:
ExpLog.candidates.add_row(
    {"candidate_id": 1, "reason_or_hypothesis": "xxx", "notes": "note1", "study_obj": tmpStudyObj1},
    allow_update=True,
)

<RowView candidates[1]>

In [12]:
ExpLog.candidates[1].df

,candidate_id,reason_or_hypothesis,notes,study_obj,OHDone,idealOverPotential
0,1,xxx,note1,<__main__.study_object object at 0x0000025457E...,<NA>,<NA>


In [13]:
process_df = pd.DataFrame(columns=["process_id", "candidate_id", "job_type", "slurmID", "status", "VASP_dir", "processNote"])
ExpLog.add_table("processes", process_df, pk="process_id")

<Table processes rows=0 pk=process_id>

In [14]:
ExpLog._tables

{'candidates': <Table candidates rows=1 pk=candidate_id>,
 'processes': <Table processes rows=0 pk=process_id>}

In [15]:
ExpLog.candidates.add_relation(
    name="processes",
    child_table="processes",
    fk_in_child="candidate_id",
)

<Table candidates rows=1 pk=candidate_id>

In [16]:
ExpLog.candidates.relations

{'processes': Relation(name='processes', child_table='processes', fk_in_child='candidate_id', parent_pk='candidate_id', local_key_in_child='process_id')}

In [17]:
ExpLog.candidates[1].processes.df

,process_id,candidate_id,job_type,slurmID,status,VASP_dir,processNote


In [18]:
ExpLog.candidates.df

,candidate_id,reason_or_hypothesis,notes,study_obj,OHDone,idealOverPotential
0,1,xxx,note1,<__main__.study_object object at 0x0000025457E...,<NA>,<NA>


In [19]:
ExpLog.candidates[1].processes.add_row(
    {"process_id": 101, "job_type": "bulk_relaxation", "slurmID": 12345, "status": "running", "VASP_dir": "/path/to/vasp1", "processNote": "bulk relaxation"},
)

<RowView processes[101]>

In [20]:
ExpLog.candidates[1].processes[101].df

,process_id,candidate_id,job_type,slurmID,status,VASP_dir,processNote
0,101,1,bulk_relaxation,12345,running,/path/to/vasp1,bulk relaxation


In [21]:
ExpLog.candidates[1].processes.add_row(
    {"process_id": 102, "job_type": "surface_relaxation", "slurmID": 12346, "status": "running", "VASP_dir": "/path/to/vasp2", "processNote": "surface relaxation"},
    allow_update=True
)

<RowView processes[102]>

In [22]:
ExpLog.candidates[1].processes.df

,process_id,candidate_id,job_type,slurmID,status,VASP_dir,processNote
0,101,1,bulk_relaxation,12345,running,/path/to/vasp1,bulk relaxation
1,102,1,surface_relaxation,12346,running,/path/to/vasp2,surface relaxation


In [23]:
ExpLog.candidates.add_row(
    {"candidate_id": 2, "reason_or_hypothesis": "yyy", "notes": "note2", "study_obj": tmpStudyObj2}
)

<RowView candidates[2]>

In [24]:
ExpLog.candidates.df

,candidate_id,reason_or_hypothesis,notes,study_obj,OHDone,idealOverPotential
0,1,xxx,note1,<__main__.study_object object at 0x0000025457E...,<NA>,<NA>
1,2,yyy,note2,<__main__.study_object object at 0x0000025457E...,<NA>,<NA>


In [25]:
ExpLog.flatten_explode()

,candidates.candidate_id,candidates.reason_or_hypothesis,candidates.notes,candidates.study_obj,candidates.OHDone,candidates.idealOverPotential,processes.process_id,processes.candidate_id,processes.job_type,processes.slurmID,processes.status,processes.VASP_dir,processes.processNote
0,1,xxx,note1,<__main__.study_object object at 0x0000025457E...,<NA>,<NA>,101,1,bulk_relaxation,12345,running,/path/to/vasp1,bulk relaxation
1,1,xxx,note1,<__main__.study_object object at 0x0000025457E...,<NA>,<NA>,102,1,surface_relaxation,12346,running,/path/to/vasp2,surface relaxation
2,2,yyy,note2,<__main__.study_object object at 0x0000025457E...,<NA>,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [26]:
ExpLog.candidates.add_row(
    {"candidate_id": 3, "reason_or_hypothesis": "zzz", "notes": "note3", "study_obj": tmpStudyObj1}
)

<RowView candidates[3]>

In [27]:
ExpLog.processes.add_row(
    {"process_id": 201, "candidate_id": 3, "job_type": "bulk_relaxation", "slurmID": 22345, "status": "completed", "VASP_dir": "/path/to/vasp3", "processNote": "bulk relaxation for candidate 2"},
)

<RowView processes[201]>

In [28]:
# tmpslurmID = ExpLog.candidates[1].processes[102].slurmID
# tmpStatus = HPC.get_status(tmpslurmID)
# ExpLog.candidates[1].processes[102].status = tmpStatus

In [29]:
ExpLog.candidates[1].processes[102].df

,process_id,candidate_id,job_type,slurmID,status,VASP_dir,processNote
1,102,1,surface_relaxation,12346,running,/path/to/vasp2,surface relaxation


In [30]:
ExpLog.flatten_explode()

,candidates.candidate_id,candidates.reason_or_hypothesis,candidates.notes,candidates.study_obj,candidates.OHDone,candidates.idealOverPotential,processes.process_id,processes.candidate_id,processes.job_type,processes.slurmID,processes.status,processes.VASP_dir,processes.processNote
0,1,xxx,note1,<__main__.study_object object at 0x0000025457E...,<NA>,<NA>,101,1,bulk_relaxation,12345,running,/path/to/vasp1,bulk relaxation
1,1,xxx,note1,<__main__.study_object object at 0x0000025457E...,<NA>,<NA>,102,1,surface_relaxation,12346,running,/path/to/vasp2,surface relaxation
2,2,yyy,note2,<__main__.study_object object at 0x0000025457E...,<NA>,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,3,zzz,note3,<__main__.study_object object at 0x0000025457E...,<NA>,<NA>,201,3,bulk_relaxation,22345,completed,/path/to/vasp3,bulk relaxation for candidate 2


In [31]:
# merge ExpLog.candidates with ExpLog.processes on candidate_id
ExpLog.candidates.merge(ExpLog.processes, left_on="candidate_id", right_on="candidate_id")

,candidate_id,reason_or_hypothesis,notes,study_obj,OHDone,idealOverPotential,process_id,job_type,slurmID,status,VASP_dir,processNote
0,1,xxx,note1,<__main__.study_object object at 0x0000025457E...,<NA>,<NA>,101,bulk_relaxation,12345,running,/path/to/vasp1,bulk relaxation
1,1,xxx,note1,<__main__.study_object object at 0x0000025457E...,<NA>,<NA>,102,surface_relaxation,12346,running,/path/to/vasp2,surface relaxation
2,3,zzz,note3,<__main__.study_object object at 0x0000025457E...,<NA>,<NA>,201,bulk_relaxation,22345,completed,/path/to/vasp3,bulk relaxation for candidate 2


In [32]:
ExpLog.candidates[1].processes[102].status = "pppppending"

In [33]:
ExpLog.candidates[1].processes.df

,process_id,candidate_id,job_type,slurmID,status,VASP_dir,processNote
0,101,1,bulk_relaxation,12345,running,/path/to/vasp1,bulk relaxation
1,102,1,surface_relaxation,12346,pppppending,/path/to/vasp2,surface relaxation
